Dylan Ross

Create a combined table (a `polars.DataFrame`) with all metadata and -omics data pre-processed, cleaned, and aligned by sample ID. This will make it super convenient to query out tables for my various data analysis needs.

## Setup

### Imports

In [1]:
import sys
import os 

# necessary to make data loading functions available
# Jupyter server should be running from src/python
sys.path.append(
    os.getcwd()
)

In [2]:
from pilot.data_import import (
    import_acetyl,
    import_global,
    import_lipids,
    import_meta,
    import_metabolites,
    import_phospho,
    import_rna,
    syn_login
)

In [3]:
from collections import defaultdict

import polars as pl
import pandas as pd

### Synapse login token/session

In [4]:
SYN = syn_login()

Welcome, dylan.ross!



### Constants

In [5]:
# Jupyter server should be running from src/python
CACHE_DIR = os.path.join(
    os.path.dirname(os.path.dirname(os.getcwd())),
    "analysis",
    "dylan",
    "_cache"
)

# metadata and combined -omics data saved to Apache arrow files
# for quick and easy loading
META_CACHE = os.path.join(CACHE_DIR, "meta.arrow")
COMBINED_CACHE = os.path.join(CACHE_DIR, "combined.arrow")

## Load Metadata and Each -Omics Dataset separately
### Metadata

In [6]:
meta = (
    pl.from_pandas(
        import_meta(SYN), 
        include_index=True
    )
    .rename({"None": "Sample"})
    # keep only a subset of the mutation data:
    # important mutations from Eisfeld paper
    # https://doi.org/10.1038/s41588-024-01929-x
    .select(
        "Sample",
        "Age",
        "Sex",
        "Race",
        "Study", 
        # for all of the mutation data, (implicit) treat "Not measured" as "WT"
        # convert the column into boolean indicating mutation state
        (pl.col("FLT3_ITD") == "Mutant"),
        # condense IDH1 and IDH2 into single IDH1 & IDH2 category
        (
            (pl.col("IDH1") == "Mutant")
            & (pl.col("IDH2") == "Mutant")
        ).alias("IDH1+IDH2"),
        (pl.col("NPM1") == "Mutant"),
        (pl.col("NRAS") == "Mutant"),
    )
    # retain only samples with "White" or "Black" race label
    .filter(pl.col("Race").is_in(["White", "Black"]))
)
meta

[syn69692583]: Downloaded to /Users/dylan.ross/.synapseCache/115/169939115/combinedBeatAML_pilotMetadata_20260317.csv


[syn64126463:beataml_waves1to4_sample_mapping.xlsx]: Found existing file at /Users/dylan.ross/.synapseCache/872/150013872/beataml_waves1to4_sample_mapping.xlsx, skipping download.
[syn64126458:1-s2.0-S1535610822003129-mmc2.xlsx]: Found existing file at /Users/dylan.ross/.synapseCache/845/150013845/1-s2.0-S1535610822003129-mmc2.xlsx, skipping download.


Sample,Age,Sex,Race,Study,FLT3_ITD,IDH1+IDH2,NPM1,NRAS
str,f64,str,str,str,bool,bool,bool,bool
"""11-00261""",74.0,"""Male""","""White""","""BeatAML""",true,false,false,false
"""11-00503""",54.0,"""Female""","""White""","""BeatAML""",true,false,true,false
"""11-00475""",65.0,"""Male""","""White""","""BeatAML""",true,false,true,false
"""12-00032""",70.0,"""Male""","""White""","""BeatAML""",true,false,false,false
"""11-00376""",49.0,"""Male""","""White""","""BeatAML""",true,false,true,false
…,…,…,…,…,…,…,…,…
"""16-01109-Bridge""",33.0,"""Male""","""Black""","""pilotStudy""",false,false,false,false
"""16-01191-Bridge""",65.0,"""Female""","""White""","""pilotStudy""",false,false,false,true
"""17-00025-Bridge""",39.0,"""Female""","""White""","""pilotStudy""",true,false,true,false


### Acetylomics

In [7]:
acetyl = (
    (_df := 
        pl.from_pandas(
            import_acetyl(SYN), 
            include_index=True
        )
    )
    .select(pl.exclude("None"))
    .transpose(
        include_header=True, 
        header_name="Feature", 
        column_names=_df["None"].to_list()
    )
    .select(
        pl.lit("Acetylomics").alias("Block"),
        pl.exclude("Block")
    )
)
acetyl

[syn69075568:ptrc_ex26_crosstab_acetyl_siteid_corrected.txt]: Found existing file at /Users/dylan.ross/.synapseCache/514/161922514/ptrc_ex26_crosstab_acetyl_siteid_corrected.txt, skipping download.
[syn25807733:Ex10_metadata.txt]: Found existing file at /Users/dylan.ross/.synapseCache/668/78060668/Ex10_metadata.txt, skipping download.
[syn68835814:PTRC_Exp26 Sample Key.xlsx]: Found existing file at /Users/dylan.ross/.synapseCache/432/161070432/PTRC_Exp26 Sample Key.xlsx, skipping download.


Block,Feature,PS88-0050,PS88-0140,PS89-0158,93-C-154,93-C-201,94-C-077,94-C-279,94-C-376,C-95-054,C-95-068,16-00494,17-00025,16-01100,17-00741,17-00881,16-00120,16-00292,16-01109,C-98-0665,C-96-117,C-97-0509,C-98-0031,C-98-0033,C-98-0454,C-98-0846,C-99-0027,C-99-0740,C-99-0901,C-99-1077,C-99-1700,C-99-2065,C-99-2136,C-00-0552,C-00-0561,C-01-0171,…,C-05-4372,C-07-2070,C-07-2900,C-08-2081,C-08-2480,C-08-3337,C-08-3381,C-08-3493,C-09-0923,C-09-1033,C-09-1074,C-09-1336,C-09-1608,C-09-1906,C-09-2182,C-09-3512,C-09-4769,C-09-5381,C-09-5462,C-10-0302,C-10-0535,C-10-0773,C-10-3265,C-10-3906,C-10-3924,C-11-0287,C-11-2295,C-11-5466,C-12-0858,C-12-1118,C-12-2943,C-12-4258,C-13-0276,16-00627,16-01191,16-00731,14-00528
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Acetylomics""","""A2M-K1176k""",null,-0.510944,1.271643,null,-1.015743,2.126205,1.190713,-2.143029,null,null,-0.608232,null,null,-0.026323,0.241584,null,null,-1.484538,null,null,null,0.313458,-1.622013,-0.126824,null,null,0.244636,0.922338,2.104951,0.880241,-2.93503,-0.428963,-1.208284,-0.624109,null,…,null,0.185746,1.686053,null,-0.66513,0.586437,null,null,0.075348,-1.505467,null,null,-0.645458,-0.303403,-0.426377,0.933879,-0.48326,1.157757,null,null,null,-1.204683,-0.866241,null,null,-1.548292,null,-0.268616,-0.677707,-2.254892,1.059783,null,-0.310935,-0.001547,null,-0.715752,null
"""Acetylomics""","""A2M-K912k""",null,-1.447282,0.277311,null,0.695807,0.545431,1.409782,0.912355,null,null,-0.652943,-0.46559,null,0.51926,0.756513,null,0.864626,0.653013,null,2.080418,1.158553,0.435211,0.38928,0.937062,1.166509,null,1.42115,0.68221,0.818989,0.454476,-0.020518,-0.219241,1.137845,0.531476,0.001479,…,null,-0.235264,null,null,null,null,null,null,-0.264098,0.695151,null,null,-0.032714,null,-0.353602,null,1.012484,0.724779,null,null,null,-0.479246,null,0.334551,null,-0.988101,null,0.50597,null,null,null,1.118524,null,-0.494212,null,0.459282,null
"""Acetylomics""","""ABCE1-K343k""",-0.119836,null,null,-0.048686,0.650037,-1.661094,-1.701827,null,-1.785763,-1.260764,-0.285838,-0.318336,0.096108,null,-0.30699,0.233756,null,null,-1.71928,-1.50428,null,0.268563,null,-0.729073,-0.39394,-0.963391,null,null,-0.715829,null,0.317464,null,-0.932423,null,-1.03373,…,-0.325684,null,null,-0.613459,-0.661235,-0.978161,0.264765,-0.649684,1.272178,0.365935,null,0.363862,null,null,null,null,null,0.21981,null,-0.688179,0.298271,null,-0.622726,-0.535396,null,0.214861,0.41977,-0.778146,-0.761498,null,null,-0.864762,-1.139557,null,-0.769978,null,-0.475381
"""Acetylomics""","""ABCE1-K431k""",-0.292101,-0.383407,-0.039556,0.379527,-0.412845,-0.911242,-0.729478,0.327699,-1.401894,-1.354356,-0.155541,0.263165,0.209144,0.416606,0.552764,-0.669246,1.047444,-1.529498,1.016596,-1.394889,-2.485997,-0.181248,-1.698226,-0.272828,-0.572122,-0.397582,-0.311457,0.722227,-0.205401,-0.274447,-0.49722,-0.031272,-0.437083,-0.161051,-0.750778,…,-0.366197,-0.957215,-1.158969,-2.268549,-1.298847,-2.113297,-1.588899,-0.776447,-1.438501,0.370127,-0.270704,-0.669587,-1.278353,-0.767274,-0.486283,-0.831399,-0.462912,-1.73836,-1.911973,-0.076643,-0.77017,-0.504609,-0.73121,-1.034007,-1.752572,0.053697,0.373132,-1.248729,-0.773947,0.193721,-1.075306,-0.946813,-1.40503,0.712604,0.151771,0.865964,-0.036184
"""Acetylomics""","""ABHD10-K69k""",-0.880812,0.040963,-0.87768,1.435475,null,null,1.21292,-1.523059,0.32947,0.512305,null,null,0.170189,1.235622,-1.203431,null,-0.028813,0.614198,null,null,1.389265,-1.051302,0.591655,null,null,0.573258,-0.686493,0.480735,0.47155,-0.609805,-1.579952,-0.788261,1.928302,2.158056,null,…,null,3.101041,3.253574,1.419707,null,null,0.89014,2.022406,-0.784229,null,1.44384,null,1.028187,0.645795,1.229874,1.755126,2.234139,null,0.899615,null,1.821945,3.772406,2.56

### Lipidomics

In [8]:
_ptrc, _pilot = import_lipids(SYN)
lipid = (
    (_df := 
        pl.concat(
            [
                pl.from_pandas(
                    _ptrc, 
                    include_index=True
                ),
                pl.from_pandas(
                    _pilot, 
                    include_index=True
                ),
            ],
            how="diagonal"
        )
    )
    .select(pl.exclude("None"))
    .transpose(
        include_header=True, 
        header_name="Feature", 
        column_names=_df["None"].to_list()
    )
    .select(
        pl.lit("Lipidomics").alias("Block"),
        pl.exclude("Block")
    )
)
lipid

[syn71896667:normalized_combat_edata_lipid.csv]: Found existing file at /Users/dylan.ross/.synapseCache/248/166162248/normalized_combat_edata_lipid.csv, skipping download.


Block,Feature,12-00032,11-00503,12-00069,13-00468,18-00103,12-00123,18-00101,12-00127,12-00145,12-00154,12-00196,12-00250,12-00268,12-00294,18-00149,12-00383,13-00016,13-00033,13-00034,13-00047,13-00059,13-00075,13-00077,13-00092,13-00123,13-00157,13-00160,13-00186,13-00195,13-00262,13-00331,13-00393,13-00149,13-00147,13-00450,…,C-08-2091,C-08-2480,C-08-3337,C-08-3381,C-08-3493,C-09-0143,C-09-0261,C-09-0923,C-09-1033,C-09-1074,C-09-1336,C-09-1608,C-09-1906,C-09-2182,C-09-3512,C-09-4769,C-09-5381,C-09-5462,C-10-0302,C-10-0535,C-10-0773,C-10-1211,C-10-3265,C-10-3429,C-10-3906,C-10-3924,C-11-0287,C-11-2295,C-11-5466,C-12-0858,C-12-1118,C-12-2943,C-12-4258,C-13-0276,16-00627,16-01191,16-00731
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Lipidomics""","""neg_LPI 20:4_[M-H]-__A""",18.435897,19.425027,18.781972,19.559651,19.229072,18.448508,18.462212,19.008276,17.816789,17.262339,17.467426,19.47388,18.541375,19.466617,17.972661,17.910608,18.993415,17.06623,19.026417,18.789276,18.182538,18.497899,19.132243,18.712868,18.039241,17.776844,18.105979,17.44444,18.57031,19.164063,18.224749,19.002001,18.50886,17.559082,17.934509,…,17.343284,17.32228,18.09385,18.24743,17.186804,18.499684,17.693258,18.077672,18.062232,17.713872,19.152533,19.131274,19.35725,19.792732,18.144107,18.847794,18.708151,17.466654,19.100675,18.646703,20.764949,18.407116,18.34649,17.201393,18.949739,17.815893,17.806289,18.41998,18.403305,18.612559,17.896223,17.811048,17.738377,18.072983,17.876114,19.132101,18.810775
"""Lipidomics""","""neg_LPG 22:6_[M-H]-__A""",15.939925,13.332694,16.258509,15.283197,15.376758,16.577574,15.052493,14.767001,14.222718,16.398119,14.580229,15.33611,15.582789,15.455396,16.647439,17.779687,15.101362,15.420531,13.334115,15.068705,11.487924,15.495652,15.474534,15.887301,16.103162,14.974481,16.0001,15.58053,13.274243,14.136811,15.689124,14.416469,16.326141,14.932304,15.580751,…,16.057472,15.539072,15.909858,16.348064,16.029266,16.033734,13.939998,15.704383,15.767129,15.287322,15.720675,17.551123,16.168995,14.918845,14.377889,16.884587,17.079708,14.215011,16.312236,14.498737,16.181298,15.295053,15.998223,14.699496,17.087171,15.056595,16.286271,15.399837,16.385283,15.616657,15.357581,14.53959,15.55933,16.920046,15.077261,15.648837,17.159722
"""Lipidomics""","""neg_LPI 20:3_[M-H]-__A""",17.03302,16.27659,15.580615,15.508878,16.293132,16.323765,17.241012,17.480965,17.282085,17.122331,16.118254,17.287977,16.577573,15.753691,16.950782,17.360527,15.800438,14.754988,16.349948,15.328972,15.710085,16.490614,15.084179,16.290234,17.52108,14.767972,16.073114,15.737933,16.196002,17.134708,15.495628,16.617408,14.670533,15.396166,16.425467,…,null,15.34704,15.368405,16.662644,14.96511,15.507413,14.400997,14.860501,15.805879,15.277685,15.539793,16.321945,17.502062,17.783228,14.919043,16.838498,16.257867,15.050656,16.020405,16.862748,19.834705,15.398115,16.207838,15.507567,16.793991,15.838628,15.612993,16.299807,15.78733,16.813625,16.083459,15.55696,16.03181,16.242012,15.510758,17.170929,16.818901
"""Lipidomics""","""neg_LPG 22:5_[M-H]-__A""",17.18434,13.769529,16.310356,16.559035,15.295408,16.291611,16.151017,14.927514,13.715198,17.136068,14.667662,15.932099,17.17814,16.371562,16.37376,15.805954,15.938042,15.261179,14.097894,15.129483,12.858515,14.915505,15.404731,15.983098,16.054884,15.875988,14.860855,14.881498,15.672194,14.906863,15.101946,15.46148,17.375415,15.131399,16.419864,…,17.129548,14.837918,17.063393,16.309982,15.903623,16.748122,15.327412,16.587611,16.912779,15.56203,17.112847,16.409388,16.175442,16.792172,15.327093,16.264481,17.435875,16.277012,17.675897,15.862881,16.742496,16.005721,16.926397,15.418413,16.67529,15.319853,16.338051,15.7259,16.169941,16.059122,16.186822,14.777519,15.323566,15.509279,

### Metabolomics

In [9]:
_ptrc, _pilot = import_metabolites(SYN)
metab = (
    (_df := 
        pl.concat(
            [
                pl.from_pandas(
                    _ptrc, 
                    include_index=True
                ),
                pl.from_pandas(
                    _pilot, 
                    include_index=True
                ),
            ],
            how="diagonal"
        )
    )
    .select(pl.exclude("None"))
    .transpose(
        include_header=True, 
        header_name="Feature", 
        column_names=_df["None"].to_list()
    )
    .select(
        pl.lit("Metabolomics").alias("Block"),
        pl.exclude("Block")
    )
)
metab

[syn71896311:rp_normalized_combat_edata_metab.csv]: Found existing file at /Users/dylan.ross/.synapseCache/125/166162125/rp_normalized_combat_edata_metab.csv, skipping download.
[syn25796769:PNNL_clinical_summary_03_03_2021.xlsx]: Found existing file at /Users/dylan.ross/.synapseCache/799/76897799/PNNL_clinical_summary_03_03_2021.xlsx, skipping download.
[syn68835814:PTRC_Exp26 Sample Key.xlsx]: Found existing file at /Users/dylan.ross/.synapseCache/432/161070432/PTRC_Exp26 Sample Key.xlsx, skipping download.


Block,Feature,12-00032,11-00503,12-00069,13-00468,18-00103,12-00123,18-00101,12-00127,12-00145,12-00154,12-00196,12-00250,12-00268,12-00294,18-00149,12-00383,13-00016,13-00033,13-00034,13-00047,13-00059,13-00075,13-00077,13-00092,13-00123,13-00157,13-00160,13-00186,13-00195,13-00262,13-00331,13-00393,13-00149,13-00147,13-00450,…,C-09-1074,C-09-1336,C-09-1608,C-09-1906,C-09-2182,C-09-3512,C-09-4769,C-09-5381,C-09-5462,C-10-0302,C-10-0535,C-10-0773,C-10-1211,C-10-3265,C-10-3429,C-10-3906,C-10-3924,C-11-0287,C-11-2295,C-11-5466,C-12-0858,C-12-1118,C-12-2943,C-12-4258,C-13-0276,16-00627,16-01191,16-00731,16-00494,17-00025,16-01100,17-00741,17-00881,16-00120,16-00292,16-01109,C-98-0665
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Metabolomics""","""rppos_3.Hydroxyhexadecanoylcar…",18.560512,20.456926,17.321128,21.317864,18.952151,17.618723,19.466776,18.478136,16.904964,19.840103,17.418109,18.516067,19.214826,20.425267,14.750338,18.797737,17.706429,15.090577,21.396441,19.2071,21.439233,15.3336,15.786503,17.069387,17.685282,18.325687,14.585494,14.723604,17.539325,19.83152,16.636314,16.612449,21.058499,17.125955,13.370101,…,17.346782,17.199605,18.997263,15.802918,16.928037,14.130853,16.533692,19.883155,15.123579,18.711861,18.370304,17.730964,16.343378,19.095459,14.035508,15.633157,17.096824,20.406942,19.191233,18.496756,16.844086,18.197451,20.079984,17.194435,16.975042,18.202975,18.060554,17.828468,19.83855,20.222337,19.482136,16.362991,20.37417,19.095295,18.580408,14.865952,22.366506
"""Metabolomics""","""rppos_5.Oxoproline""",23.962277,23.503443,23.409263,22.938007,23.494437,24.454826,24.910931,25.595707,22.867875,22.058986,22.631341,23.82986,23.228487,23.365667,24.593384,23.938365,25.438705,23.652126,23.412514,24.501844,23.098215,25.126152,24.079572,22.304367,24.382121,23.652235,22.984809,21.655368,24.959062,23.637517,23.975473,23.345713,22.216808,23.388193,24.333541,…,24.225233,23.570777,23.315419,24.741439,23.487553,24.163404,23.930632,23.773553,22.160474,25.474376,25.995752,23.040634,24.244641,23.610062,23.378467,21.990856,22.886282,25.55339,23.776649,24.145757,24.859096,24.926631,26.079559,22.005859,24.772171,23.549691,25.557336,25.353716,26.46191,25.568378,25.736419,25.673114,22.929829,24.907967,26.335303,23.470423,21.720762
"""Metabolomics""","""rppos_Acetyl.L.carnitine""",25.39362,24.277188,27.592381,29.240905,26.739821,23.035321,26.369587,26.426252,24.475636,22.735283,25.042691,26.015598,25.496206,27.549655,25.559441,26.534583,26.594596,25.540321,25.52127,27.29041,25.190277,26.635384,24.18679,24.010038,25.761064,24.878758,25.619571,26.68818,26.423349,23.455613,24.775523,26.847838,27.518276,23.247788,25.075348,…,25.992848,26.323137,25.159653,25.041541,24.604203,23.490563,25.435658,25.21754,23.14897,25.996345,25.144126,23.754881,27.10277,25.349666,24.383168,22.372676,24.943207,27.220559,25.474747,25.700509,25.59748,25.292423,26.220358,22.800699,25.930365,26.365187,28.034162,28.528256,29.156017,28.876271,28.880857,28.322491,27.146991,27.611863,29.263052,23.696274,23.24535
"""Metabolomics""","""rppos_Adenosine""",21.508335,22.36418,20.498261,24.566766,22.149919,20.014892,20.136825,21.626258,21.523075,18.798877,17.78252,23.661798,22.518133,23.221477,19.960924,21.876893,21.180283,23.805605,18.011,23.453698,22.977329,21.53343,19.882501,20.4344,22.455901,23.902172,22.004498,22.213514,21.371691,23.540279,22.020971,21.072412,23.364422,21.95915,20.890053,…,19.752544,19.351094,20.052811,20.54399,19.621588,19.873469,20.067605,19.730311,19.680589,21.79611,21.966103,17.864929,19.367325,19.302868,20.751488,17.502103,19.457855,21.466337,18.769726,20.654424,20.605302,20.587528,22.844771,20.251166,20.68268,21.096607,22.507803,23.227931,22.346316,22.748953,22.799608,22.102362,19.567887,25.151

### Proteomics

In [10]:
_ptrc, _pilot = import_global(SYN)
prot = (
    (_df := 
        pl.concat(
            [
                pl.from_pandas(
                    _ptrc, 
                    include_index=True
                ),
                pl.from_pandas(
                    _pilot, 
                    include_index=True
                ),
            ],
            how="diagonal"
        )
    )
    .select(pl.exclude("None"))
    .transpose(
        include_header=True, 
        header_name="Feature", 
        column_names=_df["None"].to_list()
    )
    .select(
        pl.lit("Proteomics").alias("Block"),
        pl.exclude("Block")
    )
)
prot

Block,Feature,11-00261,11-00503,11-00475,12-00032,11-00376,11-00378,11-00382,11-00388,11-00416,11-00465,11-00466,12-00069,12-00123,12-00127,12-00145,12-00196,13-00147,12-00294,12-00383,13-00033,13-00034,13-00075,13-00077,13-00123,13-00149,13-00157,13-00160,13-00186,13-00195,13-00226,13-00245,13-00262,13-00331,13-00393,13-00450,…,C-95-068,C-01-1665,C-09-1033,C-99-2136,C-05-1782,94-C-376,PS88-0050,C-01-2163,C-02-0350,C-04-0434,C-01-0171,PS88-0140,C-11-2295,C-99-1077,C-05-0004,C-05-3159,C-05-4372,PS89-0158,C-01-0930,C-98-0031,94-C-077,C-98-0033,C-98-0665,C-99-1700,C-02-1356,C-99-2065,14-00528-Bridge,16-00120-Bridge,16-00494-Bridge,16-00627-Bridge,16-00731-Bridge,16-01100-Bridge,16-01109-Bridge,16-01191-Bridge,17-00025-Bridge,17-00741-Bridge,17-00881-Bridge
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Proteomics""","""A1BG""",1.010275,1.193825,-0.368686,1.793044,null,-1.084904,-1.341473,-0.94186,-2.553533,-0.831849,-1.4579,0.498278,null,null,1.063305,0.190083,-0.50073,-2.501175,0.434085,-1.00815,0.462745,-0.562803,0.418758,null,-0.983038,0.523952,1.824292,0.415318,1.214632,-0.334107,0.198923,1.641109,-0.930149,-0.042924,0.777459,…,0.503249,0.744407,-0.765565,0.070162,-0.685533,0.667158,0.552594,0.999923,1.321317,0.982233,1.120692,0.07014,-2.00289,1.131453,-0.045517,-0.963941,-0.375325,1.287504,0.757538,1.161624,2.295147,1.236394,0.016356,1.109179,2.191668,1.061017,-1.765944,-0.887179,-0.45448,-0.70102,-0.116854,-0.665767,-1.844439,0.771347,-0.732169,-0.548223,-0.349491
"""Proteomics""","""A2M""",0.609935,0.59216,0.141362,-0.051768,-0.343111,0.732146,-1.077609,-0.77956,-1.418408,-0.644383,-1.205371,-0.040282,1.350584,-1.198939,0.264238,-0.319335,-0.087844,-1.315691,0.664112,-0.462416,2.134339,0.236243,1.008567,2.131375,-1.116442,0.903392,2.607579,0.318913,1.21773,-1.271589,-0.548537,3.432149,-0.990183,1.489302,0.713376,…,1.333698,-0.180129,-1.292976,1.597939,-1.667052,0.91629,-0.032401,1.021053,1.063729,1.64441,1.247292,-0.119177,-2.654269,1.033691,-0.418087,-1.09193,-0.236866,1.105128,1.036175,1.550965,2.021506,1.930924,0.726604,1.640543,1.271923,1.004152,-0.644993,-1.482487,-0.357223,-0.401315,-0.884608,-0.142247,-1.297964,-0.170057,-0.63816,0.63802,0.221334
"""Proteomics""","""AAAS""",-0.607491,-0.31949,-1.156402,-1.084748,-1.621443,-2.679351,-1.890176,-2.105576,-2.097517,-1.562096,-1.579641,-0.707875,-0.973387,-1.795937,-0.683611,-1.15703,-1.408224,-2.741277,-0.561134,-1.456652,-0.516166,-1.54825,-1.058913,-0.300366,-2.115904,-0.439687,-1.462448,-1.501422,-0.879406,-3.128997,-2.450271,-0.502961,-1.456084,-1.519986,-0.367295,…,-1.64389,-2.614208,-1.141142,-1.529728,-2.248363,-2.002754,-1.404656,-0.401436,-1.29942,-0.406512,-1.511232,-1.187713,-1.705635,-0.750425,-0.701018,-1.391553,-0.809102,-1.349431,-0.49197,-1.284759,-1.156394,-0.782666,-2.167858,-0.816144,0.253734,-1.467816,-1.892983,-1.296235,-0.408886,-3.00823,-2.662839,-1.132003,-1.50009,-2.74196,-1.093548,-0.858019,-1.690165
"""Proteomics""","""AACS""",-0.176501,0.313655,-0.609557,-0.750311,-0.073949,-1.18306,-1.25451,-1.264321,-2.277627,-1.290747,-2.207366,-0.365852,-0.03859,-1.664651,-0.46153,-1.224686,-1.551044,-2.954592,-0.303504,-0.841197,0.788282,-1.329812,-0.078728,0.871793,-1.981292,-0.062389,-1.034471,-1.342467,0.222108,-3.033531,-2.152721,-0.491726,-1.520839,-1.66603,0.145144,…,-1.698733,-1.654561,-0.842278,-0.092,0.018038,-0.412975,-1.383157,-1.220393,0.266542,-0.240316,-1.964346,-0.944109,-2.00945,-1.581639,-0.445194,-0.923957,-0.569268,0.178282,-0.742533,-1.530399,-0.63022,-1.777491,-1.735419,-0.298714,-0.111104,-0.711214,-0.334524,-1.617261,0.869457,-0.951102,-1.529419,1.017977,-3.263164,-0.343664,0.061794,-0.270661,-0.613057
"""Proteomics""","""AAGAB""",-0.580698,-0.4766,-0.341627,-1.124299,-1.454701,-2.9

### Phosphoproteomics

In [11]:
_ptrc, _pilot = import_phospho(SYN)
phospho = (
    (_df := 
        pl.concat(
            [
                pl.from_pandas(
                    _ptrc, 
                    include_index=True
                ),
                pl.from_pandas(
                    _pilot, 
                    include_index=True
                ),
            ],
            how="diagonal"
        )
    )
    .select(pl.exclude("None"))
    .transpose(
        include_header=True, 
        header_name="Feature", 
        column_names=_df["None"].to_list()
    )
    .select(
        pl.lit("Phosphoproteomics").alias("Block"),
        pl.exclude("Block")
    )
)
phospho

Block,Feature,11-00261,11-00503,11-00475,12-00032,11-00376,11-00378,11-00382,11-00388,11-00416,11-00465,11-00466,12-00069,12-00123,12-00127,12-00145,12-00196,13-00147,12-00294,12-00383,13-00033,13-00034,13-00075,13-00077,13-00123,13-00149,13-00157,13-00160,13-00186,13-00195,13-00226,13-00245,13-00262,13-00331,13-00393,13-00450,…,C-95-068,C-01-1665,C-09-1033,C-99-2136,C-05-1782,94-C-376,PS88-0050,C-01-2163,C-02-0350,C-04-0434,C-01-0171,PS88-0140,C-11-2295,C-99-1077,C-05-0004,C-05-3159,C-05-4372,PS89-0158,C-01-0930,C-98-0031,94-C-077,C-98-0033,C-98-0665,C-99-1700,C-02-1356,C-99-2065,14-00528-Bridge,16-00120-Bridge,16-00494-Bridge,16-00627-Bridge,16-00731-Bridge,16-01100-Bridge,16-01109-Bridge,16-01191-Bridge,17-00025-Bridge,17-00741-Bridge,17-00881-Bridge
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Phosphoproteomics""","""AAAS-S495s""",-0.228,-2.62,-0.111,null,null,null,-5.3,-3.58,-1.22,-1.25,-2.61,-5.23,null,0.0147,-0.892,-0.333,null,-1.59,null,null,-0.799,-0.17,-2.65,-1.5,-1.52,-4.13,-5.43,-3.17,-4.64,null,null,-0.259,-0.475,null,-0.534,…,-0.070358,-2.546159,-0.678956,-1.377696,-0.980433,-0.965089,0.253306,-0.104865,-0.410867,0.727484,-0.412822,-0.190408,-0.619896,-0.740228,0.130107,-1.06414,0.048576,0.037059,0.244974,-1.270726,-0.237886,0.814452,-2.727012,-0.16619,0.748594,-1.585919,-0.049049,-0.859636,0.357416,-0.980801,-1.391123,-0.274277,-0.033924,-3.043648,0.196599,-0.252866,-2.054192
"""Phosphoproteomics""","""AAK1-S20s""",0.693856,0.408721,-0.94158,-0.424579,0.703566,-1.732005,-0.635543,-0.537547,null,-0.744995,null,0.106242,0.397976,-0.890643,-0.55065,-1.309272,-0.519009,-1.895894,-0.125291,-0.299182,0.696533,-0.607979,0.165682,1.623686,-0.875183,-0.07325,-0.519544,-0.712439,0.482483,-1.931273,null,0.004459,-1.139513,-1.161068,-0.810165,…,null,-1.258214,null,0.279889,-0.810108,0.486404,null,0.843728,null,-0.090613,0.303768,0.751027,null,-1.09226,null,-0.260491,0.312562,1.288768,null,-0.413065,null,1.216551,-1.40153,0.008384,null,0.430239,-0.010062,-0.509899,null,null,-0.113599,null,-1.787005,null,0.559486,-0.197032,-0.677475
"""Phosphoproteomics""","""AAK1-S637s""",0.043272,2.317306,-0.775356,-6.389542,0.280428,-1.847803,-1.414204,-4.939151,-4.214155,-0.183061,-2.664992,-2.2941,-0.759245,-1.912305,-1.433181,-0.993864,-3.062853,-2.583468,-2.854584,-2.508529,0.61582,-0.394834,-1.486149,2.31749,-2.784627,-1.440515,-5.81749,-1.328161,-0.192808,-3.606629,null,0.253963,-1.524541,-4.716482,-1.396878,…,-2.508596,-3.459761,-2.866562,1.359772,-2.477557,-0.716246,-1.142248,0.565292,-0.527545,-4.32048,0.077927,0.061208,-2.722415,-2.353024,0.990825,-1.126458,-2.095451,0.731432,0.54202,-1.365001,-1.495029,-2.731382,-3.686851,-1.365721,-3.381137,-1.083227,-0.922248,-0.206884,-0.056312,-2.943713,-3.887491,0.014164,-3.874337,-3.538961,-0.148168,-2.16566,-2.468972
"""Phosphoproteomics""","""AAK1-S670sT674tS678s""",null,0.095059,-0.5958,-0.741351,null,-3.017537,null,-1.305877,-2.450587,-1.844609,null,-0.374305,-1.376778,-2.542745,null,null,null,-3.307811,-0.866881,-2.77231,-0.391756,null,-0.886811,1.119391,null,null,null,-2.381676,-0.036482,null,null,-0.643511,null,-3.935296,null,…,-2.556301,-3.289489,-1.689746,-0.42027,-1.838956,-1.401806,-1.230694,0.141009,0.242613,-1.061358,-0.079802,-0.558943,-0.831081,-2.023619,1.396926,-0.582053,-1.694131,0.115586,-0.252434,-0.593775,-2.432905,0.369233,-3.92299,-1.416649,-4.480351,-0.603058,-1.313252,-0.790995,-0.412861,-2.769204,-2.648549,1.243047,-2.989417,-1.971776,0.181735,-1.709711,-1.761734
"""Phosphoproteomics""","""AAK1-S678s""",2.024118,1.650225,0.630539,0.371569,null,1.134727,1.320309,1.000382,-0.49035,1.169107,null,1.421212,null,0.255917,null,null,0.667989,-0.236031,2.501417,0.530456,3.25502,0.698188,0.504447,1.764667,0.423262,0.747749

### Transcriptomics

In [12]:
def rename_redundant_columns(df):
    cols = []
    count = defaultdict(int)
    for column in df.columns:
        if df.columns.tolist().count(column) > 1:
            cols.append(f"{column}__{count[column] + 1}")
            count[column] += 1
        else:
            cols.append(column)
    df.columns = cols
    return df

In [13]:
_ptrc, _pilot = import_rna(SYN)
rna = (
    (_df := 
        pl.concat(
            [
                pl.from_pandas(
                    # rename duplicated columns with suffixes __1, __2, etc.
                    rename_redundant_columns(_ptrc.loc[:, pd.notnull(_ptrc.columns)]), 
                    include_index=True
                ),
                pl.from_pandas(
                    # rename duplicated columns with suffixes __1, __2, etc.
                    rename_redundant_columns(_pilot.loc[:, pd.notnull(_pilot.columns)]),
                    include_index=True
                ),
            ],
            how="diagonal"
        )
    )
    .select(pl.exclude("None"))
    .transpose(
        include_header=True, 
        header_name="Feature", 
        column_names=_df["None"].to_list()
    )
    .select(
        pl.lit("Transcriptomics").alias("Block"),
        pl.exclude("Block")
    )
)
rna

[syn64126462:beataml_waves1to4_counts_dbgap.txt]: Found existing file at /Users/dylan.ross/.synapseCache/867/150013867/beataml_waves1to4_counts_dbgap.txt, skipping download.
[syn68820229:RNAseq expression data.txt]: Found existing file at /Users/dylan.ross/.synapseCache/317/160991317/RNAseq expression data.txt, skipping download.
[syn68820228:PilotYR_1_2_metadata_083024_withRNA DN.csv]: Found existing file at /Users/dylan.ross/.synapseCache/316/160991316/PilotYR_1_2_metadata_083024_withRNA DN.csv, skipping download.
[syn64126463:beataml_waves1to4_sample_mapping.xlsx]: Found existing file at /Users/dylan.ross/.synapseCache/872/150013872/beataml_waves1to4_sample_mapping.xlsx, skipping download.


Block,Feature,12-00023,12-00051,12-00066,12-00150,12-00211,12-00258,12-00294,12-00372,12-00423,12-00426,13-00007,13-00016,13-00028,13-00034,13-00098,13-00118,13-00123,13-00126,13-00138,13-00145,13-00146,13-00147,13-00149,13-00150,13-00157,13-00160,13-00163,13-00165,13-00166,13-00195,13-00202,13-00204,13-00226,13-00232,13-00245,…,C-98-1247,C-99-0027,C-99-0740,C-99-0901,C-99-1077,C-99-1483,C-99-1700,C-01-1599,C-04-0434,C-05-0004,C-05-0319,C-05-3159,C-08-2091,C-09-0143,C-09-3512,C-09-5462,C-98-0665,C-00-0552,C-01-2163,C-05-3084,C-10-0535,C-11-0287,93-C-201,94-C-376,C-07-2900,C-98-0033,C-98-0846,C-99-2065,C-01-1665,C-12-2943,C-07-2070,C-08-3493,C-09-4769,C-05-0927,C-05-0664,C-00-1828,C-98-0031
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Transcriptomics""","""TSPAN6""",0.103669,0.65572,0.043229,0.640248,0.0,0.0,0.0,0.426835,0.186139,0.098532,0.034152,0.009039,0.0,0.0,0.0,0.065038,0.0,0.248792,0.046093,0.071955,0.026398,0.476653,0.094712,0.666195,0.0,0.554116,0.022239,0.123715,0.100291,0.044208,0.0,0.0,0.013633,0.014323,0.014627,…,0.482489,0.047955,0.615752,0.0,0.024344,0.310879,0.314319,0.438559,0.0,0.250363,0.625636,0.0,0.0,0.214669,0.112238,0.098647,0.984077,0.090034,0.0,1.643708,1.376733,0.151713,0.213582,0.080825,0.0,0.0,0.200571,0.433158,1.88484,0.0,0.119746,0.0,0.153299,0.971454,0.0,0.622786,0.0
"""Transcriptomics""","""DPM1""",42.2949,39.501528,32.369528,41.013938,35.796944,43.962098,46.677819,34.83103,32.095636,30.882779,40.314684,49.831257,40.344293,49.212928,58.122603,36.59215,34.97804,37.681882,43.419913,37.320002,61.777688,58.162187,43.795988,46.951723,46.680031,47.185714,47.359576,39.041338,45.514755,43.27632,47.601183,56.160941,44.860049,48.803604,39.187933,…,21.130105,30.060728,46.764976,53.782709,58.78328,56.380604,53.621586,38.368226,36.425492,23.062369,48.735105,38.679187,75.639395,57.286305,65.152223,52.858465,25.148159,28.760345,21.800403,45.259315,53.095656,31.440215,68.715484,38.242273,58.871256,40.784846,74.059334,38.895909,49.343057,27.938921,48.739766,49.055112,48.311296,25.859969,59.031227,51.696317,64.137597
"""Transcriptomics""","""SCYL3""",4.637513,4.24828,2.324006,3.786193,9.688506,3.872938,10.532277,9.17871,4.036118,9.534841,4.617313,6.43063,5.942431,4.268526,3.808655,6.775408,3.797257,9.24937,2.374742,4.934254,2.028864,4.51285,5.827211,6.197588,9.966381,1.751508,3.529134,8.09201,10.99305,4.973334,6.203946,3.582807,4.804644,5.295974,5.731609,…,6.844946,1.776001,3.61,2.927941,5.874823,3.80684,2.597266,2.927874,4.623899,2.691919,3.737155,5.711506,4.956968,3.679198,5.9937,13.096241,4.161802,3.549523,4.93876,9.496555,3.78003,8.337406,4.359002,5.950507,3.592602,7.346956,9.724426,7.096905,3.46894,4.478246,3.540668,1.51846,1.785633,4.323122,0.0,3.348115,2.358715
"""Transcriptomics""","""FIRRM""",2.649945,0.925847,4.31657,3.969572,4.997403,3.365956,7.942868,4.710336,4.282219,3.773954,7.912645,7.149849,5.157167,2.62771,9.808667,5.297318,3.492156,3.908501,6.745936,1.859633,3.910797,2.974716,11.033525,6.533829,3.34584,4.893394,7.525448,4.980718,3.215416,3.952697,4.162647,3.405613,5.511921,5.173987,5.298076,…,5.856955,1.24406,1.964662,3.465588,3.884587,3.80421,2.128287,4.217538,3.77882,3.899228,3.499829,3.922926,6.018329,6.724833,3.779721,12.288715,6.088482,12.317475,2.666249,2.901687,6.549641,8.143883,2.509051,3.659507,4.782109,6.957766,5.857764,15.962179,1.396348,4.173299,1.523938,5.927285,4.389623,6.181564,0.0,7.112927,2.424203
"""Transcriptomics""","""FGR""",432.529394,6.684816,484.090112,19.653993,54.110998,577.286719,143.423666,90.838567,102.958435,30.411213,7.775512,81.421577,31.741366,332.745742,4.491205,108.657407,608.548608,69.796145,197.999416,32.983411,1.621948,65.418489,30.433106,122.357421,187.640341,267.174279,248.623088,47.353929,23.143883,439.9

## Combine the (-Omics Data) Tables
(metadata table stays separate with samples as rows, to enable easier subsetting of samples based on metadata)

In [14]:
combined = (
    pl.concat(
        [
            acetyl,
            lipid,
            metab,
            prot,
            phospho,
            rna
        ],
        how="diagonal",
    )
    # keep only the sample columns that are included in the filtered metadata table
    .select(
        ["Block", "Feature"] + meta["Sample"].to_list()
    )
)
combined

Block,Feature,11-00261,11-00503,11-00475,12-00032,11-00376,11-00378,11-00382,11-00388,11-00416,11-00465,11-00466,12-00123,12-00127,12-00145,12-00196,12-00294,12-00383,13-00033,13-00034,13-00075,13-00077,13-00123,13-00149,13-00157,13-00160,13-00186,13-00195,13-00226,13-00262,13-00331,13-00450,13-00468,14-00495,13-00558,13-00581,…,C-95-068,C-01-1665,C-09-1033,C-99-2136,C-05-1782,94-C-376,PS88-0050,C-01-2163,C-02-0350,C-04-0434,C-01-0171,PS88-0140,C-11-2295,C-99-1077,C-05-0004,C-05-3159,C-05-4372,PS89-0158,C-01-0930,C-98-0031,94-C-077,C-98-0033,C-98-0665,C-99-1700,C-02-1356,C-99-2065,14-00528-Bridge,16-00120-Bridge,16-00494-Bridge,16-00627-Bridge,16-00731-Bridge,16-01100-Bridge,16-01109-Bridge,16-01191-Bridge,17-00025-Bridge,17-00741-Bridge,17-00881-Bridge
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Acetylomics""","""A2M-K1176k""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,0.330127,-1.505467,-0.428963,null,-2.143029,null,-0.497628,-1.37027,-1.787643,null,-0.510944,null,2.104951,-0.382382,-0.304169,null,1.271643,null,0.313458,2.126205,-1.622013,null,0.880241,null,-2.93503,null,null,null,null,null,null,null,null,null,null,null
"""Acetylomics""","""A2M-K912k""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,0.38503,0.695151,-0.219241,null,0.912355,null,null,0.890893,0.79203,0.001479,-1.447282,null,0.818989,-0.03042,null,null,0.277311,null,0.435211,0.545431,0.38928,null,0.454476,null,-0.020518,null,null,null,null,null,null,null,null,null,null,null
"""Acetylomics""","""ABCE1-K343k""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,-1.260764,-0.196359,0.365935,null,-0.641335,null,-0.119836,0.099939,0.551293,null,-1.03373,null,0.41977,-0.715829,-0.281123,null,-0.325684,null,0.658935,0.268563,-1.661094,null,-1.71928,null,0.094878,0.317464,null,null,null,null,null,null,null,null,null,null,null
"""Acetylomics""","""ABCE1-K431k""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,-1.354356,0.582544,0.370127,-0.031272,-0.086364,0.327699,-0.292101,-1.60355,-0.159053,-0.481542,-0.750778,-0.383407,0.373132,-0.205401,-0.688846,-0.799921,-0.366197,-0.039556,0.701408,-0.181248,-0.911242,-1.698226,1.016596,-0.274447,-0.361193,-0.49722,null,null,null,null,null,null,null,null,null,null,null
"""Acetylomics""","""ABHD10-K69k""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,0.512305,-0.641897,null,-0.788261,null,-1.523059,-0.880812,-0.147372,null,-0.379555,null,0.040963,1.265502,0.47155,null,-1.411947,null,-0.87768,-0.02174,-1.051302,null,0.591655,null,-0.609805,0.167879,-1.579952,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Transcriptomics""","""CEP43__2""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.371185,null,null,0.0,null,null,0.087719,0.249785,0.155027,0.099639,null,0.155454,0.156892,0.038594,0.185828,0.141952,0.126895,0.206017,0.094288,0.241965,…,0.077266,0.146562,0.540351,null,null,0.038757,null,0.069963,0.088275,0.338344,0.088325,null,null,0.350199,0.1

## Cache the Assembled Tables

In [15]:
meta.write_ipc(META_CACHE)

In [16]:
combined.write_ipc(COMBINED_CACHE)